Trying to create a personal 3D stitiching algorithm

In [2]:
import tifffile
import pandas as pd
from skimage.measure import regionprops, regionprops_table
import napari
import numpy as np

def properties_mask(image):

    # Get the properties of every mask in the image
    props = regionprops(image)

    data = []

    # For every mask found, get the label, centeroid, boundingbox, and volume
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area
        
        # Pre-compute coordinate set for fast IOU calculations later
        coords_2d = prop.coords[:, 1:]
        coords_set = set(map(tuple, coords_2d))

        data.append(
            {
                "label": label,
                "z": centroid[0],
                "y": centroid[1],
                "x": centroid[2],
                "bounding_box": bounding_box,
                "volume": volume,
                "coords_2d": coords_2d,
                "coords_set": coords_set,
            }
        )

    # Make a pandas data frame from the data
    df_props = pd.DataFrame(data)

    return df_props


def properties_channel(mask, image, type=None):

    # Get the properties of the mask for the channel you specified (image)
    props_channel = regionprops_table(
        mask, intensity_image=image, properties=["label", "mean_intensity"]
    )

    # Make this into a pandas dataframe and rename some variables to the name of the channel for later merging
    props_channel = pd.DataFrame(props_channel).rename(
        columns={
            "label": "label",
            "mean_intensity": f"raw_{type}",
        }
    )

    return props_channel

def make_unique_mask(mask):
    """Makes a fresh mask from a already stitched mask"""
    unique_mask = []
    number = 1
    for z in range(mask.shape[0]):
        # go over every unique number in the mask and create a new slice with a new unique number 1 higher
        slice = mask[z]
        unique_slice = mask[z].copy()
        for val in range(1, slice.max() + 1):
            unique_slice[slice == val] = number
            number += 1
        unique_mask.append(unique_slice)
    unique_mask = np.stack(unique_mask, axis=0)

    return unique_mask

Dual nuclei image

In [27]:
# generate mask
import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils

image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)

nuclei = image[:, 0:1]

nuclei = np.max(image, axis=1)

model = utils.load_model(r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei")

mask = utils.segment(nuclei, model)

tifffile.imwrite(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif",
    mask,
)

Using CUDA for processing.
loaded custom model: dual_nuclei


In [72]:
image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)
crc_image = image[:,0]
wt_image = image[:,1]
mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif"
)
# print(mask.shape)
# # to the mask, add pixel on the left side with value 0 for broken image
# mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode='constant', constant_values=0)
# print(mask.shape)

unique_mask = make_unique_mask(mask)

props_mask = properties_mask(unique_mask)
props_wt = properties_channel(unique_mask, wt_image, type="wt")
props_crc = properties_channel(unique_mask, crc_image, type="crc")

# Merge all the dataframes together on the label
df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")

In [7]:
def calculate_iou(idx1, idx2):
    """
    Calculate overlap metric for cell matching.
    Returns 1 if the smaller cell is completely contained in the larger cell.
    Otherwise returns intersection over union (standard IOU).
    """
    set1 = coords_sets[idx1]
    set2 = coords_sets[idx2]

    # Check if either set is empty
    len1, len2 = len(set1), len(set2)
    if len2 == 0:
        return 0

    # Calculate intersection (overlapping pixels)
    intersection = len(set1 & set2)

    if intersection == 0:
        return 0  # No overlap = different cells

    # If the smaller cell is completely contained in the larger cell, return 1
    min_len = min(len1, len2)
    if intersection == min_len:
        return 1.0  # One cell completely covers the other

    # Otherwise, use standard IOU
    union = len1 + len2 - intersection
    return intersection / union

In [214]:
import time

total_start_time = time.time()

print("Initializing data structures...")
# Reset index to ensure sequential 0-based indexing
df_final = df_final.reset_index(drop=True)

df_final["3d_label"] = -1  # Use -1 to indicate unassigned
mask_3D = np.zeros_like(unique_mask)
coords_sets = df_final["coords_set"].to_dict()

z_groups = {}
for idx, z_val in df_final["z"].items():
    z_int = int(round(z_val))
    if z_int not in z_groups:
        z_groups[z_int] = []
    z_groups[z_int].append(idx)

z_array = df_final["z"].values
x_array = df_final["x"].values
y_array = df_final["y"].values
labels_array = np.full(len(df_final), -1, dtype=np.int32)

init_time = time.time() - total_start_time
print(f"Initialization completed in {init_time:.3f}s")

print("Starting main stitching process...")
stitching_start = time.time()


def stitch_forward(idx, current_label):
    """Simplified recursive stitching - idx is now both pandas index and array position"""
    current_z = int(round(z_array[idx]))

    # Assign the current label
    labels_array[idx] = current_label

    # Fast lookup: get only indices in the next slice
    next_z = current_z + 1
    if next_z not in z_groups:
        return

    candidates = z_groups[next_z]

    # Filter unassigned candidates
    unassigned_candidates = [c for c in candidates if labels_array[c] == -1]

    if len(unassigned_candidates) == 0:
        return

    # Vectorized distance calculation
    unassigned_array = np.array(unassigned_candidates)
    distances = np.sqrt(
        (x_array[unassigned_array] - x_array[idx]) ** 2
        + (y_array[unassigned_array] - y_array[idx]) ** 2
    )

    # Find the closest candidate
    closest_idx = np.argmin(distances)
    closest_distance = distances[closest_idx]

    # Only proceed if the closest cell is within threshold
    if closest_distance > 10:
        return

    # Get the closest candidate's index
    closest_candidate = unassigned_array[closest_idx]

    # Check IOU with the closest candidate only
    iou = calculate_iou(idx, closest_candidate)
    if iou > 0.3:
        # Recursively stitch this cell forward
        stitch_forward(closest_candidate, current_label)


unique_label = 0
for idx in range(len(df_final)):
    if labels_array[idx] == -1:
        unique_label += 1
        stitch_forward(idx, unique_label)

# Copy results back to dataframe
df_final["3d_label"] = labels_array

stitching_time = time.time() - stitching_start
print(f"Main stitching completed in {stitching_time:.3f}s")
print(f"Total unique 3D cells found: {unique_label}")

print("Creating final 3D mask...")
mask_creation_start = time.time()
# Vectorized mask creation
for idx, row in df_final.iterrows():
    z = int(row["z"])
    label_3d = labels_array[idx]
    mask_3D[z][unique_mask[z] == row["label"]] = label_3d

mask_creation_time = time.time() - mask_creation_start
print(f"Final 3D mask created in {mask_creation_time:.3f}s")

Initializing data structures...
Initialization completed in 0.009s
Starting main stitching process...
Main stitching completed in 0.059s
Total unique 3D cells found: 328
Creating final 3D mask...
Final 3D mask created in 0.080s


In [259]:
df_final

,label,z,y,x,bounding_box,volume,coords_2d,coords_set,raw_wt,raw_crc,3d_label
0,1,4.0,157.123853,256.073394,"(4, 147, 244, 5, 169, 269)",436.0,"[[147, 251], [147, 252], [147, 253], [147, 254...","{(152, 246), (152, 255), (152, 264), (164, 247...",118.791284,447.660550,1
1,2,4.0,174.197309,254.883408,"(4, 168, 245, 5, 183, 266)",223.0,"[[168, 251], [168, 252], [168, 253], [168, 254...","{(174, 249), (175, 248), (169, 262), (174, 258...",118.910314,491.479821,2
2,3,5.0,115.375776,227.354037,"(5, 107, 216, 6, 124, 241)",322.0,"[[107, 223], [107, 224], [107, 225], [107, 226...","{(112, 219), (112, 228), (123, 228), (115, 224...",114.139752,295.590062,3
3,4,5.0,133.435897,264.235043,"(5, 124, 255, 6, 143, 275)",234.0,"[[124, 258], [125, 256], [125, 257], [125, 258...","{(132, 268), (137, 264), (128, 261), (137, 273...",115.209402,289.927350,4
4,5,5.0,156.303675,257.425532,"(5, 145, 244, 6, 168, 272)",517.0,"[[145, 256], [145, 257], [145, 258], [145, 259...","{(152, 246), (152, 255), (152, 264), (164, 247...",124.578337,555.831721,1
...,...,...,...,...,...,...,...,...,...,...,...
1900,1901,59.0,260.649606,170.937008,"(59, 250, 160, 60, 272, 182)",254.0,"[[250, 162], [250, 163], [250, 164], [250, 165...","{(259, 165), (259, 174), (261, 171), (252, 168...",130.700787,107.007874,324
1901,1902,59.0,265.350515,153.582474,"(59, 259, 145, 60, 273, 162)",194.0,"[[259, 150], [259, 151], [259, 152], [259, 153...","{(261, 153), (266, 149), (267, 148), (266, 158...",137.288660,107.113402,319
1902,1903,60.0,150.567164,255.014925,"(60, 145, 249, 61, 157, 263)",134.0,"[[145, 253], [145, 254], [145, 255], [145, 256...","{(150, 258), (152, 255), (155, 254), (148, 260...",130.805970,107.440299,269
1903,1904,60.0,225.052632,174.347368,"(60, 221, 168, 61, 230, 182)",95.0,"[[221, 171], [221, 172], [221, 173], [221, 174...","{(223, 178), (224, 177), (229, 173), (228, 177...",132.778947,107.715789,322


In [ ]:
def break_score(values, i):
    # Calculate intensity-based anomalies by getting a score that resembles high-low-high intensity value patterns
    # Calculate ratio between slice i and i+1, and i+1 and i+2
    ratio_1 = values[i - 1] / (values[i])
    ratio_2 = values[i] / (values[i + 1])
    # divide this again, now high low high gets higher score then other patterns
    ratio_total = ratio_1 / ratio_2
    return ratio_total

def point_line_distance(z_positions, x_positions, y_positions):
    # Fit line to first 3 points
    z_fit = np.array(z_positions[:3])
    x_fit = np.array(x_positions[:3])
    y_fit = np.array(y_positions[:3])

    # Calculate line parameters
    A = np.vstack([z_fit, np.ones(len(z_fit))]).T
    m_x, c_x = np.linalg.lstsq(A, x_fit, rcond=None)[0]
    m_y, c_y = np.linalg.lstsq(A, y_fit, rcond=None)[0]

    # Calculate distances for all points
    distances = []
    for i in range(len(z_positions)):
        z_val = z_positions[i]
        x_line = m_x * z_val + c_x
        y_line = m_y * z_val + c_y
        distance = np.sqrt((x_positions[i] - x_line) ** 2 + (y_positions[i] - y_line) ** 2)
        distances.append(distance)
    return distances

def slope_change(distances):
    # First derivative (slope between consecutive points)
    first_deriv = np.diff(distances)

    # Second derivative (change in slope)
    second_deriv = np.diff(first_deriv)

    # Pad to match original length
    second_deriv_padded = np.concatenate([[0], second_deriv, [0]])

    return second_deriv_padded

In [239]:
# Create a new dataframe grouped by 3D label

def calculate_3d_properties(df_3d_props):
    # Optional: add number of slices per 3D cell
    df_3d_props["num_slices"] = df_3d_props["labels_2d"].apply(len)
    df_3d_props["volumes_break_score"] = df_3d_props["num_slices"].apply(lambda x: [0]*x)
    df_3d_props["final_break_score"] = df_3d_props["num_slices"].apply(lambda x: [0] * x)
    df_3d_props["break_score_up"] = df_3d_props["num_slices"].apply(lambda x: [0] * x)
    df_3d_props["break_score_down"] = df_3d_props["num_slices"].apply(lambda x: [0] * x)
    df_3d_props["intensity_break_score"] = df_3d_props["num_slices"].apply(
        lambda x: [0] * x
    )  # Initialize break score column which is an array of len num_slices
    # For each cell, get the max intensity between wt and crc at each slice
    df_3d_props["nuclei_intensity"] = df_3d_props.apply(
        lambda row: [max(wt, crc) for wt, crc in zip(row["raw_wt"], row["raw_crc"])], axis=1
    )

    # itterate over every row, look at the volumes, find cells where the pattern is big small big
    for idx, row in df_3d_props.iterrows():
        volumes = row["volumes"]
        intensities = row["nuclei_intensity"]
        if len(volumes) < 5:
            continue  # At least 5 slices to break something

        for i in range(1, len(volumes) - 1):
            df_3d_props.at[idx, "volumes_break_score"][i] = break_score(volumes, i)
            df_3d_props.at[idx, "intensity_break_score"][i] = break_score(intensities, i)
            # multiply both scores to get a final score

        # Calculate a line based on first 3 points and calculate distance from this line for other points
        z_positions = row["z_positions"]
        x_positions = row["x_centroids"]
        y_positions = row["y_centroids"]
        line_distance_up = point_line_distance(z_positions, x_positions, y_positions)
        # # get delta between consecutive points
        # delta_line_distance_up = abs(np.diff(line_distance_up))
        # break_score_up = [0] + [score / 1 for score in delta_line_distance_up]
        break_score_up = abs(slope_change(line_distance_up))
        df_3d_props.at[idx, "break_score_up"] = break_score_up

        # Calculate a line based on last 3 points and calculate distance from this line for other points
        line_distance_down = point_line_distance(
            z_positions[::-1], x_positions[::-1], y_positions[::-1])[::-1]
        # delta_line_distance_down = abs(np.diff(line_distance_down))
        # break_score_down = [0] + [score / 1 for score in delta_line_distance_down]
        break_score_down = abs(slope_change(line_distance_down))
        df_3d_props.at[idx, "break_score_down"] = break_score_down

        # calculate final break score by multiplying all scores
        final_scores = []
        for v_score, i_score, up_score, down_score in zip(
            df_3d_props.at[idx, "volumes_break_score"],
            df_3d_props.at[idx, "intensity_break_score"],
            df_3d_props.at[idx, "break_score_up"],
            df_3d_props.at[idx, "break_score_down"],
        ):
            final_score = np.max([up_score, down_score]) * v_score * i_score
            final_scores.append(final_score)
        df_3d_props.at[idx, "final_break_score"] = final_scores

    return df_3d_props

In [247]:
# Method 2: Access the actual list value
cell = 107
intensity = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell][
    "nuclei_intensity"
].values[0])
intensity_break = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["intensity_break_score"].values[0])
volume = df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["volumes"].values[0]
volume_break = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["volumes_break_score"].values[0])
break_score_up = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["break_score_up"].values[0])
break_score_down = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["break_score_down"].values[0])
final_score = (df_3d_props_fixed[df_3d_props_fixed["3d_label"] == cell]["final_break_score"].values[0])

# create df of this and display
df_cell_detail = pd.DataFrame(
    {
        "slice": range(len(volume)),
        "volume": volume,
        "nuclei_intensity": intensity,
        "intensity_break_score": intensity_break,
        "volume_break_score": volume_break,
        "break_score_up": break_score_up,
        "break_score_down": break_score_down,
        "final_break_score": final_score,
    }
)



display(df_cell_detail)

,slice,volume,nuclei_intensity,intensity_break_score,volume_break_score,break_score_up,break_score_down,final_break_score
0,0,37.0,347.729730,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,72.0,448.902778,0.920369,0.742284,0.085530,0.231759,0.158332
2,2,104.0,533.365385,0.859041,0.918639,0.524956,0.061637,0.414268
3,3,138.0,544.391304,1.020184,0.742701,0.407544,0.049273,0.308793
4,4,136.0,566.860294,0.889672,1.052011,0.454525,0.000455,0.425411
5,5,141.0,525.134752,1.080593,0.656707,0.063801,0.195280,0.138577
6,6,96.0,525.687500,0.866896,1.560547,0.616380,0.446688,0.833858
7,7,102.0,456.196078,0.939010,1.051903,0.021734,0.458610,0.452991
8,8,114.0,371.745614,0.909487,1.216528,0.554598,0.943876,1.044320
9,9,155.0,275.509677,1.206982,0.569407,2.343001,0.387637,1.610260


In [257]:
threshold = 2.5
# create copy of df final for manipulation
data = df_final.copy()

def make_data_frame_3d(df_final):
    uncut_data = (
        df_final.groupby("3d_label")
        .agg(
            {
                "label": list,
                "z": list,
                "y": list,
                "x": list,
                "volume": list,
                "raw_wt": list,
                "raw_crc": list,
            }
        )
        .reset_index()
    )

    uncut_data.columns = [
        "3d_label",
        "labels_2d",
        "z_positions",
        "y_centroids",
        "x_centroids",
        "volumes",
        "raw_wt",
        "raw_crc",
    ]

    return uncut_data


def stitch_loop(df_final):
    # Make 3D dataframe from current df_final state
    uncut_data = make_data_frame_3d(df_final)

    # Calculate break scores
    df_3d_props = calculate_3d_properties(uncut_data)

    # Get the maximum existing 3D label to continue numbering
    new_label_counter = df_3d_props["3d_label"].max() + 1

    breaks_found = 0  # Track if any breaks were made

    cut_cells = []
    # Break cells at slices with break score above threshold
    for idx, row in df_3d_props.iterrows():
        break_scores = row["final_break_score"]
        labels_2d = row["labels_2d"]

        # Only consider breaks in middle slices (skip first 3 and last 3)
        # Need at least 3 slices on each side for valid detection
        if len(labels_2d) < 7:  # Too short to have valid breaks
            continue

        # Mask out first 3 and last 3 positions
        masked_scores = np.array(break_scores, dtype=float)
        masked_scores[:3] = 0  # First 3 slices get score 0
        masked_scores[-3:] = 0  # Last 3 slices get score 0

        # If no breaks above threshold, skip this cell
        if max(masked_scores) < threshold:
            continue

        breaks_found += 1

        # Find slice with highest break score
        max_break = max(masked_scores)
        break_position = [
            i for i, score in enumerate(masked_scores) if score == max_break
        ][0]

        # Get the 2D labels
        labels_2d = row["labels_2d"]

        # All slices after and including break get a new 3D label
        for i in range(break_position, len(labels_2d)):
            label_2d = labels_2d[i]
            df_final.loc[df_final["label"] == label_2d, "3d_label"] = new_label_counter

        cut_cells.append(row["3d_label"])
        cut_cells.append(new_label_counter)
        new_label_counter += 1

    return df_3d_props, df_final, breaks_found , cut_cells


# loop until no more breaks are found
iteration = 0
all_cut_cells = []
while True:
    iteration += 1
    print(f"Stitching iteration {iteration}...")
    
    df_3d_props_fixed, data, breaks_found, cut_cells = stitch_loop(data)
    all_cut_cells.extend(cut_cells)
    # Stop if no breaks were made this iteration
    if breaks_found == 0:
        print("No more breaks found, stopping.")
        break

    print(f"Breaks made this iteration: {breaks_found}")

print(all_cut_cells)
# Create final 3D mask
mask_3d_fixed = np.zeros_like(mask_3D)
for idx, row in df_3d_props_fixed.iterrows():
    z_positions = row["z_positions"]
    labels_2d = row["labels_2d"]
    label_3d = row["3d_label"]
    for z, label_2d in zip(z_positions, labels_2d):
        mask_3d_fixed[int(z)][unique_mask[int(z)] == label_2d] = label_3d

print(f"Final number of 3D cells: {np.max(mask_3d_fixed)}")

uncut_data = make_data_frame_3d(df_final)
# Create a new 3D mask based on df_3d_props
mask_3d = np.zeros_like(mask_3D)
for idx, row in uncut_data.iterrows():
    z_positions = row["z_positions"]
    labels_2d = row["labels_2d"]
    label_3d = row["3d_label"]
    for z, label_2d in zip(z_positions, labels_2d):
        mask_3d[int(z)][unique_mask[int(z)] == label_2d] = label_3d

print(np.max(mask_3d))

mask_3d_fixed = np.zeros_like(mask_3D)
for idx, row in df_3d_props_fixed.iterrows():
    z_positions = row["z_positions"]
    labels_2d = row["labels_2d"]
    label_3d = row["3d_label"]
    for z, label_2d in zip(z_positions, labels_2d):
        mask_3d_fixed[int(z)][unique_mask[int(z)] == label_2d] = label_3d

print(np.max(mask_3d_fixed))

# make a mask that has only the cells that were cut
mask_3d_cuts = np.zeros_like(mask_3D)
# Copy only pixels from mask_3d_fixed where the value is in all_cut_cells
for cell_label in all_cut_cells:
    mask_3d_cuts[mask_3d_fixed == cell_label] = cell_label

Stitching iteration 1...
Breaks made this iteration: 35
Stitching iteration 2...
Breaks made this iteration: 8
Stitching iteration 3...
Breaks made this iteration: 2
Stitching iteration 4...
No more breaks found, stopping.
[67, 329, 82, 330, 93, 331, 104, 332, 106, 333, 118, 334, 119, 335, 120, 336, 121, 337, 122, 338, 123, 339, 125, 340, 129, 341, 130, 342, 136, 343, 138, 344, 139, 345, 142, 346, 143, 347, 144, 348, 148, 349, 149, 350, 153, 351, 155, 352, 157, 353, 160, 354, 167, 355, 176, 356, 189, 357, 199, 358, 202, 359, 226, 360, 232, 361, 248, 362, 272, 363, 106, 364, 129, 365, 157, 366, 331, 367, 333, 368, 338, 369, 344, 370, 352, 371, 369, 372, 370, 373]
Final number of 3D cells: 373
328
373


In [258]:
import napari

viewer = napari.Viewer(ndisplay=3)

# Create a shared color dictionary for specific labels
import numpy as np

# Generate colors for labels 0-max_label
max_label = max(mask_3d.max(), mask_3d_fixed.max())
np.random.seed(42)  # For reproducible colors
color_dict = {}
for i in range(max_label + 1):
    # Generate bright colors with full alpha
    color = np.random.rand(3)  # RGB only
    color = (
        color / color.max()
    )  # Normalize so at least one channel is at max brightness
    color_dict[i] = np.append(color, 0.8)  # Add full alpha

color_dict[0] = np.array([0, 0, 0, 0])  # Background transparent

# Apply to both layers
viewer.add_labels(
    mask_3d, name="3D_stitched_mask", scale=(2.5, 0.64, 0.64), colormap=color_dict
)
viewer.add_labels(
    mask_3d_fixed,
    name="3D_stitched_mask_fixed",
    scale=(2.5, 0.64, 0.64),
    colormap=color_dict,
)
viewer.add_labels(
    mask_3d_cuts,
    name="3D_stitched_mask_cuts",
    scale=(2.5, 0.64, 0.64),
    colormap=color_dict,
)

viewer.add_labels(
    unique_mask, name="original_mask", scale=(2.5, 0.64, 0.64), visible=False
)
viewer.add_image(
    crc_image,
    name="crc_image",
    scale=(2.5, 0.64, 0.64),
    blending="additive",
    visible=False,
)
viewer.add_image(
    wt_image,
    name="wt_image",
    scale=(2.5, 0.64, 0.64),
    blending="additive",
    visible=False,
)

c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\6331823\AppData\Local\Programs\Python\Python311\Lib\site-packages\napari\utils\colormaps\colormap.py:447: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


RuntimeError: sequence argument must have length equal to input rank

<Image layer 'wt_image' at 0x23c1120ae90>

In [22]:
def fit_3d_line(centroids):
    """
    Fit a 3D line through centroids using least squares.
    Returns the line direction vector and a point on the line.

    centroids: list of (z, y, x) tuples
    """
    if len(centroids) < 2:
        return None, None

    centroids = np.array(centroids)

    # Center the points
    centroid_mean = centroids.mean(axis=0)
    centered = centroids - centroid_mean

    # SVD to find the best-fit line direction
    _, _, Vt = np.linalg.svd(centered)
    direction = Vt[0]  # First principal component

    # Normalize direction
    direction = direction / np.linalg.norm(direction)

    return direction, centroid_mean


def distance_to_line(point, line_point, line_direction):
    """
    Calculate perpendicular distance from point to 3D line.
    """
    point = np.array(point)
    line_point = np.array(line_point)
    line_direction = np.array(line_direction)

    # Vector from line point to the point
    vec = point - line_point

    # Project onto the line direction
    proj_length = np.dot(vec, line_direction)
    proj = proj_length * line_direction

    # Perpendicular component
    perp = vec - proj

    return np.linalg.norm(perp)


def stitch_forward_with_trajectory(
    idx, current_label, trajectory_points=None, debug=False
):
    """Recursive stitching with 3D trajectory validation"""
    if trajectory_points is None:
        trajectory_points = []

    current_z = int(round(z_array[idx]))

    # Add current point to trajectory
    current_centroid = (z_array[idx], y_array[idx], x_array[idx])
    trajectory_points.append(current_centroid)

    # Assign the current label
    labels_array[idx] = current_label

    if debug:
        print(f"Cell {current_label}: z={current_z}, centroid={current_centroid}")

    # Fast lookup: get only indices in the next slice
    next_z = current_z + 1
    if next_z not in z_groups:
        if debug:
            print(f"  → End of stack (no z={next_z})")
        return

    candidates = z_groups[next_z]

    # Filter unassigned candidates
    unassigned_candidates = [c for c in candidates if labels_array[c] == -1]

    if len(unassigned_candidates) == 0:
        if debug:
            print(f"  → No unassigned candidates in z={next_z}")
        return

    # Vectorized distance calculation
    unassigned_array = np.array(unassigned_candidates)
    distances = np.sqrt(
        (x_array[unassigned_array] - x_array[idx]) ** 2
        + (y_array[unassigned_array] - y_array[idx]) ** 2
    )

    # Find the closest candidate
    closest_idx = np.argmin(distances)
    closest_distance = distances[closest_idx]

    if debug:
        print(f"  → Closest distance: {closest_distance:.2f}")

    # Only proceed if the closest cell is within threshold
    if closest_distance > 10:
        if debug:
            print(f"  → Distance > 10 threshold, stopping")
        return

    # Get the closest candidate's index
    closest_candidate = unassigned_array[closest_idx]

    # Check IOU with the closest candidate only
    iou = calculate_iou(idx, closest_candidate)
    if debug:
        print(f"  → IOU with next cell: {iou:.3f}")

    if iou < 0.3:
        if debug:
            print(f"  → IOU < 0.3 threshold, stopping")
        return

    # Validate trajectory deviation
    if len(trajectory_points) >= 2:
        # Fit line through all points so far
        direction, line_point = fit_3d_line(trajectory_points)
        if direction is not None:
            # Check if next candidate deviates too much from trajectory
            next_centroid = (
                z_array[closest_candidate],
                y_array[closest_candidate],
                x_array[closest_candidate],
            )
            deviation = distance_to_line(next_centroid, line_point, direction)

            if debug:
                print(f"  → Trajectory deviation: {deviation:.3f}")

            # If deviation is too large, start a new cell
            if deviation > 2.0:  # Increased threshold to be less strict
                if debug:
                    print(f"  → Deviation > 5.0 threshold, stopping")
                return

    # Recursively stitch this cell forward
    if debug:
        print(f"  → Stitching to cell at z={next_z}")
    stitch_forward_with_trajectory(
        closest_candidate, current_label, trajectory_points, debug=debug
    )

import time

total_start_time = time.time()

print("Initializing data structures...")
# Reset index to ensure sequential 0-based indexing
df_final = df_final.reset_index(drop=True)

df_final["3d_label"] = -1  # Use -1 to indicate unassigned
mask_3D = np.zeros_like(unique_mask)
coords_sets = df_final["coords_set"].to_dict()

z_groups = {}
for idx, z_val in df_final["z"].items():
    z_int = int(round(z_val))
    if z_int not in z_groups:
        z_groups[z_int] = []
    z_groups[z_int].append(idx)

z_array = df_final["z"].values
x_array = df_final["x"].values
y_array = df_final["y"].values
labels_array = np.full(len(df_final), -1, dtype=np.int32)

# Main stitching loop with debug output for first few cells
print("Starting main stitching process...")
stitching_start = time.time()

unique_label = 0
for idx in range(len(df_final)):
    if labels_array[idx] == -1:
        unique_label += 1
        # Enable debug for first 3 cells to see what's happening
        debug_mode = unique_label <= 3
        stitch_forward_with_trajectory(idx, unique_label, debug=debug_mode)

# Copy results back to dataframe
df_final["3d_label"] = labels_array

stitching_time = time.time() - stitching_start
print(f"Main stitching completed in {stitching_time:.3f}s")
print(f"Total unique 3D cells found: {unique_label}")

mask_creation_start = time.time()
# Vectorized mask creation
for idx, row in df_final.iterrows():
    z = int(row["z"])
    label_3d = labels_array[idx]
    mask_3D[z][unique_mask[z] == row["label"]] = label_3d

Initializing data structures...
Starting main stitching process...
Cell 1: z=4, centroid=(4.0, 157.12385321100916, 256.07339449541286)
  → Closest distance: 1.58
  → IOU with next cell: 0.822
  → Stitching to cell at z=5
Cell 1: z=5, centroid=(5.0, 156.3036750483559, 257.4255319148936)
  → Closest distance: 0.61
  → IOU with next cell: 1.000
  → Trajectory deviation: 0.525
  → Stitching to cell at z=6
Cell 1: z=6, centroid=(6.0, 155.94630872483222, 257.91498881431767)
  → Closest distance: 21.00
  → Distance > 10 threshold, stopping
Cell 2: z=4, centroid=(4.0, 174.19730941704037, 254.88340807174887)
  → Closest distance: 4.75
  → IOU with next cell: 0.438
  → Stitching to cell at z=5
Cell 2: z=5, centroid=(5.0, 173.73214285714286, 250.15178571428572)
  → Closest distance: 14.37
  → Distance > 10 threshold, stopping
Cell 3: z=5, centroid=(5.0, 115.37577639751552, 227.35403726708074)
  → Closest distance: 2.70
  → IOU with next cell: 0.676
  → Stitching to cell at z=6
Cell 3: z=6, centro

In [23]:
import napari
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(mask_3D, name="3D_stitched_mask", scale = (2.5,0.64,0.64))
viewer.add_labels(unique_mask, name="original_mask", scale = (2.5,0.64,0.64), visible=False)
viewer.add_image(crc_image, name="crc_image", scale = (2.5,0.64,0.64), blending='additive')
viewer.add_image(wt_image, name="wt_image", scale = (2.5,0.64,0.64), blending='additive')

RuntimeError: sequence argument must have length equal to input rank

<Image layer 'wt_image' at 0x23b26d20b90>